**Objetivo:** Construir un chatbot que responda preguntas en base a un dataset de ventas usando RAG (Retrieval-Augmented Generation) con el modelo Mistral via API.
## ¿Qué es cada concepto?
- **Corpus:** El conjunto de datos/documentos que le damos al modelo como base de conocimiento. En este caso, nuestro dataset de ventas.
- **RAG (Retrieval-Augmented Generation):** Técnica que combina búsqueda de información relevante + generación de texto. En vez de 'entrenar' el modelo, le pasamos contexto relevante en cada pregunta.
- **Agentes:** Componentes que toman decisiones y ejecutan acciones (en este caso, el agente decide qué parte del corpus buscar).
- **Mistral:** Modelo de lenguaje de código abierto (francés). Usamos su API, similar a como funciona ChatGPT pero con Mistral.

## PASO 1 — Instalación de librerías

In [6]:
!pip install -q pandas langchain langchain-experimental tabulate langchain-mistralai

## PASO 2 — Configurar la API Key de Mistral

In [38]:
from google.colab import userdata
import os

# Leemos la API key desde Secrets de Colab
MISTRAL_API_KEY = userdata.get('ventas')

# La guardamos también como variable de entorno (buena práctica)
os.environ['MISTRAL_API_KEY'] = MISTRAL_API_KEY

print('✅ API Key cargada correctamente')

✅ API Key cargada correctamente


## PASO 3 — Cargar dataset original


In [24]:

import pandas as pd

df_raw = pd.read_csv("sales_data_sample.csv", encoding="latin-1")

print("Filas:", df_raw.shape[0])
print("Columnas:", df_raw.shape[1])

df_raw.head()

Filas: 2823
Columnas: 25


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,Motorcycles,95,S10_1678,Toys4GrownUps.com,6265557265,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,Motorcycles,95,S10_1678,Corporate Gift Ideas Co.,6505551386,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


In [25]:
# PASO 4 — Exploración inicial
print("=== TIPOS DE DATOS ===")
print(df_raw.dtypes)

print("\n=== VALORES NULOS ===")
print(df_raw.isnull().sum())

df_raw.describe()

=== TIPOS DE DATOS ===
ORDERNUMBER           int64
QUANTITYORDERED       int64
PRICEEACH           float64
ORDERLINENUMBER       int64
SALES               float64
ORDERDATE            object
STATUS               object
QTR_ID                int64
MONTH_ID              int64
YEAR_ID               int64
PRODUCTLINE          object
MSRP                  int64
PRODUCTCODE          object
CUSTOMERNAME         object
PHONE                object
ADDRESSLINE1         object
ADDRESSLINE2         object
CITY                 object
STATE                object
POSTALCODE           object
COUNTRY              object
TERRITORY            object
CONTACTLASTNAME      object
CONTACTFIRSTNAME     object
DEALSIZE             object
dtype: object

=== VALORES NULOS ===
ORDERNUMBER            0
QUANTITYORDERED        0
PRICEEACH              0
ORDERLINENUMBER        0
SALES                  0
ORDERDATE              0
STATUS                 0
QTR_ID                 0
MONTH_ID               0
YEAR_ID        

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,QTR_ID,MONTH_ID,YEAR_ID,MSRP
count,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.000000,2823.00000,2823.000000
mean,10258.725115,35.092809,83.658544,6.466171,3553.889072,2.717676,7.092455,2003.81509,100.715551
std,92.085478,9.741443,20.174277,4.225841,1841.865106,1.203878,3.656633,0.69967,40.187912
min,10100.000000,6.000000,26.880000,1.000000,482.130000,1.000000,1.000000,2003.00000,33.000000
25%,10180.000000,27.000000,68.860000,3.000000,2203.430000,2.000000,4.000000,2003.00000,68.000000
50%,10262.000000,35.000000,95.700000,6.000000,3184.800000,3.000000,8.000000,2004.00000,99.000000
75%,10333.500000,43.000000,100.000000,9.000000,4508.000000,4.000000,11.000000,2004.00000,124.000000
max,10425.000000,97.000000,100.000000,18.000000,14082.800000,4.000000,12.000000,2005.00000,214.000000


## PASO 4 — Normalización del dataset

**¿Por qué normalizar?** El dataset tiene varios problemas que afectan la calidad del análisis:
- Fechas en formato texto (no reconocidas como fecha)
- Valores nulos en columnas como ADDRESSLINE2, STATE, POSTALCODE
- Caracteres especiales corruptos (encoding incorrecto)
- Columnas redundantes o irrelevantes para el chatbot
- Tipos de datos incorrectos

In [26]:


df = pd.read_csv("sales_data_sample.csv", encoding="latin-1")

# Convertir fecha
df["ORDERDATE"] = pd.to_datetime(df["ORDERDATE"], format="%m/%d/%Y %H:%M")

# Rellenar valores nulos
df["ADDRESSLINE2"] = df["ADDRESSLINE2"].fillna("N/A")
df["STATE"] = df["STATE"].fillna("N/A")
df["POSTALCODE"] = df["POSTALCODE"].fillna("N/A")
df["TERRITORY"] = df["TERRITORY"].fillna("N/A")

# Eliminar duplicados
df = df.drop_duplicates()

# Crear columnas útiles para análisis
df["YEAR"] = df["ORDERDATE"].dt.year
df["MONTH"] = df["ORDERDATE"].dt.month
df["DAY"] = df["ORDERDATE"].dt.day

# Normalizar textos
columnas_texto = [
    "STATUS", "PRODUCTLINE", "CUSTOMERNAME", "COUNTRY",
    "CITY", "DEALSIZE", "TERRITORY"
]

for col in columnas_texto:
    df[col] = df[col].astype(str).str.strip()

# Guardar dataset normalizado
df.to_csv("sales_data_normalizado.csv", index=False, encoding="utf-8")

print("✅ Dataset normalizado correctamente")
print("Dimensiones finales:", df.shape)

df.head()

✅ Dataset normalizado correctamente
Dimensiones finales: (2823, 28)


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE,YEAR,MONTH,DAY
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,1,2,2003,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,N/A,NYC,NY,10022,USA,N/A,Yu,Kwai,Small,2003,2,24
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,N/A,Reims,N/A,51100,France,EMEA,Henriot,Paul,Small,2003,5,7
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,N/A,Paris,N/A,75508,France,EMEA,Da Cunha,Daniel,Medium,2003,7,1
3,10145,45,83.26,6,3746.70,2003-08-25,Shipped,3,8,2003,Motorcycles,95,S10_1678,Toys4GrownUps.com,6265557265,78934 Hillside Dr.,N/A,Pasadena,CA,90003,USA,N/A,Young,Julie,Medium,2003,8,25
4,10159,49,100.00,14,5205.27,2003-10-10,Shipped,4,10,2003,Motorcycles,95,S10_1678,Corporate Gift Ideas Co.,6505551386,7734 Strong St.,N/A,San Francisco,CA,N/A,USA,N/A,Brown,Julie,Medium,2003,10,10


## PASO 6 — Verificación de la normalización


In [27]:

print("=== NULOS RESTANTES ===")
print(df.isnull().sum())

print("\n=== TIPOS DE DATOS FINALES ===")
print(df.dtypes)

=== NULOS RESTANTES ===
ORDERNUMBER         0
QUANTITYORDERED     0
PRICEEACH           0
ORDERLINENUMBER     0
SALES               0
ORDERDATE           0
STATUS              0
QTR_ID              0
MONTH_ID            0
YEAR_ID             0
PRODUCTLINE         0
MSRP                0
PRODUCTCODE         0
CUSTOMERNAME        0
PHONE               0
ADDRESSLINE1        0
ADDRESSLINE2        0
CITY                0
STATE               0
POSTALCODE          0
COUNTRY             0
TERRITORY           0
CONTACTLASTNAME     0
CONTACTFIRSTNAME    0
DEALSIZE            0
YEAR                0
MONTH               0
DAY                 0
dtype: int64

=== TIPOS DE DATOS FINALES ===
ORDERNUMBER                  int64
QUANTITYORDERED              int64
PRICEEACH                  float64
ORDERLINENUMBER              int64
SALES                      float64
ORDERDATE           datetime64[ns]
STATUS                      object
QTR_ID                       int64
MONTH_ID                     int64


## PASO 7 — Importar herramientas para el agente

In [29]:


from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_mistralai import ChatMistralAI

## PASO 8 — Crear modelo Mistral

In [30]:


llm = ChatMistralAI(
    model="open-mistral-7b",
    temperature=0
)

print("✅ Modelo Mistral cargado correctamente")

✅ Modelo Mistral cargado correctamente


## PASO 9 — Crear instrucciones del agente

In [31]:


prefix = """
Sos un asistente experto en análisis de datos de ventas.

Tu tarea es responder preguntas mirando únicamente el corpus cargado en el DataFrame.
El corpus corresponde a un dataset de ventas ya normalizado.

Respondé siempre en español, de forma clara y concisa.
Si la respuesta requiere cálculos, usá pandas sobre el DataFrame.
Si la información no existe en el corpus, indicá que no se puede responder con los datos disponibles.
"""

## PASO 10 — Crear agente que mira el corpus

In [34]:


agent = create_pandas_dataframe_agent(
    llm,
    df,
    prefix=prefix,
    verbose=False,
    allow_dangerous_code=True
)

print("✅ Agente creado correctamente")

✅ Agente creado correctamente


## PASO 11 — Prueba

In [35]:


pregunta1 = "¿De qué trata este corpus?"
respuesta1 = agent.invoke(pregunta1)

print(respuesta1["output"])

**
Este corpus es un **dataset de ventas detallado** que registra transacciones comerciales con los siguientes elementos clave:

- **Datos de pedidos**: Número de pedido (`ORDERNUMBER`), líneas de pedido (`ORDERLINENUMBER`), cantidades (`QUANTITYORDERED`), precios unitarios (`PRICEEACH`) y ventas totales (`SALES`).
- **Información temporal**: Fecha del pedido (`ORDERDATE`), trimestre (`QTR_ID`), mes (`MONTH_ID`/`MONTH`), año (`YEAR_ID`/`YEAR`) y día (`DAY`).
- **Productos**: Línea de producto (`PRODUCTLINE`), código (`PRODUCTCODE`), precio de lista (`MSRP`).
- **Clientes**: Nombre (`CUSTOMERNAME`), contacto (`CONTACTLASTNAME`/`CONTACTFIRSTNAME`), dirección (`ADDRESSLINE1`/`ADDRESSLINE2`), ciudad (`CITY`), estado (`STATE`), código postal (`POSTALCODE`), país (`COUNTRY`), teléfono (`PHONE`) y territorio (`TERRITORY`).
- **Estado del pedido**: `STATUS` (ej: "Shipped").
- **Tamaño del trato**: `DEALSIZE` (ej: "Small", "Medium").

**En resumen**: Es un registro histórico de ventas por clien

In [39]:
pregunta2 = "¿Cuáles son las líneas de producto disponibles?"
respuesta2 = agent.invoke(pregunta2)

print(respuesta2["output"])

Las líneas de producto disponibles en el dataset son:
- Motorcycles
- Classic Cars
- Trucks and Buses
- Vintage Cars
- Planes
- Ships
- Trains


###PASO 1 — Instalar dependencias para RAG

In [57]:
!pip install -q --upgrade langchain langchain-core langchain-community langchain-mistralai chromadb sentence-transformers opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-http

### Imports necesarios


In [ ]:
import os
import pandas as pd

from google.colab import userdata

from langchain_core.documents import Document
from langchain_mistralai import ChatMistralAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

PASO 3 — Cargar archivo normalizado

In [45]:
df = pd.read_csv("/content/sales_data_normalizado.csv", encoding="utf-8")

print("Dataset normalizado cargado correctamente")
print("Dimensiones:", df.shape)

df.head()

Dataset normalizado cargado correctamente
Dimensiones: (2823, 25)


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,1,2,2003,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,2003-08-25,Shipped,3,8,2003,Motorcycles,95,S10_1678,Toys4GrownUps.com,6265557265,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,2003-10-10,Shipped,4,10,2003,Motorcycles,95,S10_1678,Corporate Gift Ideas Co.,6505551386,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


PASO 4 — Cargar API Key de Mistral

In [46]:
MISTRAL_API_KEY = userdata.get("ventas")

os.environ["MISTRAL_API_KEY"] = MISTRAL_API_KEY

print("✅ API Key cargada correctamente")

✅ API Key cargada correctamente


PASO 5 — Convertir el DataFrame normalizado en documentos

In [51]:
documents = []

# Ensure 'ORDERDATE' is datetime and create 'YEAR', 'MONTH', 'DAY' if missing
if 'YEAR' not in df.columns:
    df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'], format="%Y-%m-%d")
    df["YEAR"] = df["ORDERDATE"].dt.year
    df["MONTH"] = df["ORDERDATE"].dt.month
    df["DAY"] = df["ORDERDATE"].dt.day

for index, row in df.iterrows():
    texto = f"""
    Registro de venta número {index + 1}.

    Número de pedido: {row['ORDERNUMBER']}
    Cantidad ordenada: {row['QUANTITYORDERED']}
    Precio unitario: {row['PRICEEACH']}
    Venta total: {row['SALES']}
    Fecha del pedido: {row['ORDERDATE']}
    Estado del pedido: {row['STATUS']}

    Línea de producto: {row['PRODUCTLINE']}
    Código de producto: {row['PRODUCTCODE']}
    Precio sugerido: {row['MSRP']}
    Tamaño del trato: {row['DEALSIZE']}

    Cliente: {row['CUSTOMERNAME']}
    País: {row['COUNTRY']}
    Ciudad: {row['CITY']}
    Territorio: {row['TERRITORY']}

    Año: {row['YEAR']}
    Mes: {row['MONTH']}
    Día: {row['DAY']}
    """

    documento = Document(
        page_content=texto,
        metadata={
            "indice": index,
            "pais": row["COUNTRY"],
            "producto": row["PRODUCTLINE"],
            "cliente": row["CUSTOMERNAME"],
            "anio": int(row["YEAR"])
        }
    )

    documents.append(documento)

print("✅ Documentos creados:", len(documents))
print(documents[0].page_content)

✅ Documentos creados: 2823

    Registro de venta número 1.

    Número de pedido: 10107
    Cantidad ordenada: 30
    Precio unitario: 95.7
    Venta total: 2871.0
    Fecha del pedido: 2003-02-24 00:00:00
    Estado del pedido: Shipped

    Línea de producto: Motorcycles
    Código de producto: S10_1678
    Precio sugerido: 95
    Tamaño del trato: Small

    Cliente: Land of Toys Inc.
    País: USA
    Ciudad: NYC
    Territorio: nan

    Año: 2003
    Mes: 2
    Día: 24
    


PASO 6 — Crear embeddings

In [49]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embeddings creados correctamente")

/tmp/ipykernel_47383/2800208726.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embeddings creados correctamente


PASO 7 — Crear base vectorial con Chroma